In [14]:
from openai import OpenAI
import pandas as pd
import minsearch


## Ingestion

In [15]:
df = pd.read_csv('../data/cleaned_data.csv')
df.head()

,id,task,category,difficulty,duration_estimate,framework_name,reasoning,instructions,tags
0,1,Write a 2-page project summary,work,medium,45,Time Blocking,Time Blocking helps allocate a clear writing w...,"Block 45 minutes, outline key points, write su...",work;writing;planning
1,2,Organize pantry shelves,home,medium,40,Task Decomposition,Breaking the pantry into sections makes the ta...,"Empty one shelf, clean it, categorize items, r...",home;organization
2,3,Practice 20-minute mobility routine,fitness,easy,20,Daily Rituals,Mobility benefits from consistent daily habits.,"Warm up joints, perform hip circles, shoulder ...",fitness;mobility;routine
3,4,Review study notes for chapter 3,study,easy,25,Spaced Repetition,Reviewing notes at intervals improves retention.,"Read notes, highlight key ideas, summarize in ...",study;memory
4,5,Write a blog outline,creative,medium,30,Mind Mapping,Mind Mapping helps structure ideas visually.,"Create central topic, branch subtopics, add su...",creative;writing


In [16]:
documents = []

for _, row in df.iterrows():
    documents.append({
        "id": str(row["id"]),
        "task": row["task"],
        "category": row["category"],
        "difficulty": row["difficulty"],
        "duration_estimate": str(row["duration_estimate"]),
        "framework_name": row["framework_name"],
        "reasoning": row["reasoning"],
        "instructions": row["instructions"],
        "tags": row["tags"],
    })



In [17]:
index = minsearch.Index(['task', 'category', 'difficulty',
       'framework_name', 'reasoning', 'instructions', 'tags'],
        keyword_fields=["id", 'duration_estimate'])


In [18]:
index.fit(documents)

print(documents[0])

{'id': '1', 'task': 'Write a 2-page project summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '45', 'framework_name': 'Time Blocking', 'reasoning': 'Time Blocking helps allocate a clear writing window.', 'instructions': 'Block 45 minutes, outline key points, write summary, revise.', 'tags': 'work;writing;planning'}


In [19]:
q = "How can I organize my workspace efficiently using a structured method?"

## RAG flow

In [20]:
client = OpenAI()

def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [21]:
index.search(q, num_results=10)

[{'id': '185',
  'task': 'Clean bedroom nightstand',
  'category': 'home',
  'difficulty': 'easy',
  'duration_estimate': '15',
  'framework_name': 'Task Batching',
  'reasoning': 'Batching cleaning tasks increases efficiency.',
  'instructions': 'Wipe surface, organize items, clean drawer.',
  'tags': 'home;cleaning'},
 {'id': '191',
  'task': 'Organize digital notes',
  'category': 'home',
  'difficulty': 'medium',
  'duration_estimate': '35',
  'framework_name': 'GTD',
  'reasoning': 'GTD helps categorize digital clutter.',
  'instructions': 'Sort notes, create folders, delete old items.',
  'tags': 'home;digital'},
 {'id': '110',
  'task': 'Clean laundry area',
  'category': 'home',
  'difficulty': 'medium',
  'duration_estimate': '25',
  'framework_name': 'Task Batching',
  'reasoning': 'Batching cleaning tasks increases efficiency.',
  'instructions': 'Wipe surfaces, organize detergents, clean floor.',
  'tags': 'home;cleaning'},
 {'id': '210',
  'task': 'Clean living room shelve

In [22]:


prompt_template = """
You're a productivity advisor. Answer the QUESTION based on the CONTEXT from our productivity tasks dataset.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
task: {task}
category: {category}
difficulty: {difficulty}
duration_estimate: {duration_estimate}
instructions: {instructions}
reasoning: {reasoning}
tags: {tags}
""".strip()

def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context += entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt



In [23]:
search_results = search(q)
prompt = build_prompt(q, search_results)


In [24]:

def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content


In [25]:
def rag(query, model='gpt-4o-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, model=model)
    return answer

In [26]:
question = "How can I organize my workspace efficiently using a structured method?"
answer = rag(question)
print(answer)

To organize your workspace efficiently using a structured method, you can follow these steps:

1. **Clear and Clean**: Begin by clearing your desk completely and wiping down the surfaces to ensure a clean starting point.

2. **Sort Items**: Categorize the items on your desk into groups based on their function and importance. For example, separate essential items from non-essential ones.

3. **Categorize and Organize**: Utilize the GTD (Getting Things Done) methodology, which helps in categorizing items. Place only essentials back on the desk, such as tools and supplies you use frequently.

4. **Storage Solutions**: Label containers and storage solutions to keep similar items together. This can reduce clutter and make it easier to find what you need.

5. **Break Down Tasks**: If your workspace involves multiple sections (like drawers or shelves), consider breaking these down into smaller tasks to avoid feeling overwhelmed, similar to how organizing a bathroom cabinet or laundry supplies

In [27]:
query = "What key points should I include in my outline for the project summary?"
ans = rag(query)
print(ans)



For your project summary outline, consider including the following key points:

1. **Introduction**: Provide a brief overview of the project.
2. **Achievements**: List the key accomplishments and milestones reached during the project.
3. **Challenges**: Identify any issues encountered throughout the project.
4. **Proposed Solutions**: Suggest actionable solutions to the challenges faced.
5. **Next Steps**: Outline the proposed next steps for the project moving forward.
6. **Deadlines**: Set clear deadlines for the proposed next steps or solutions.
7. **Conclusion**: Summarize the project's overall impact and importance. 

Using the SMART structure will enhance clarity in your summary.


## Retrieval evaluation

In [28]:
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')
df_question.head()

,id,question
0,1,What key points should I include in my outline...
1,1,How can I effectively manage my time during th...
2,1,What strategies can I use to ensure my summary...
3,1,How should I approach the revision process aft...
4,1,Are there any specific writing techniques that...


In [29]:
ground_truth = df_question.to_dict(orient='records')

In [30]:

ground_truth[0]

{'id': 1,
 'question': 'What key points should I include in my outline for the project summary?'}

In [31]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [32]:
def minsearch_search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [33]:
from tqdm.auto import tqdm

In [34]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = str(q['id'])
        results = search_function(q)
        relevance = [str(d['id']) == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }


In [35]:
print(ground_truth[0])

{'id': 1, 'question': 'What key points should I include in my outline for the project summary?'}


In [36]:
print(minsearch_search(ground_truth[0]["question"]))

[{'id': '1', 'task': 'Write a 2-page project summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '45', 'framework_name': 'Time Blocking', 'reasoning': 'Time Blocking helps allocate a clear writing window.', 'instructions': 'Block 45 minutes, outline key points, write summary, revise.', 'tags': 'work;writing;planning'}, {'id': '217', 'task': 'Write project summary draft', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '30', 'framework_name': 'SMART Goals', 'reasoning': 'SMART structure improves clarity in summaries.', 'instructions': 'Write intro, list achievements, propose next steps.', 'tags': 'work;writing'}, {'id': '242', 'task': 'Write project improvement summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': '30', 'framework_name': 'SMART Goals', 'reasoning': 'SMART structure improves clarity in improvement summaries.', 'instructions': 'List issues, propose solutions, set deadlines.', 'tags': 'work;analysis'}, {'id': '19

In [37]:

evaluate(ground_truth, lambda q: minsearch_search(q['question']))

  0%|          | 0/1250 [00:00<?, ?it/s]

{'hit_rate': 0.8072, 'mrr': 0.6059936507936513}

## Finding the best parameters

In [38]:
df_validation = df_question[:100]
df_testing = df_question[100:]

In [39]:
import random

def simple_optimize(param_ranges, objective_function, n_iterations=10):
    best_params = None
    best_score = float('-inf')  # Assuming we're minimizing. Use float('-inf') if maximizing.

    for _ in range(n_iterations):
        # Generate random parameters
        current_params = {}
        for param, (min_val, max_val) in param_ranges.items():
            if isinstance(min_val, int) and isinstance(max_val, int):
                current_params[param] = random.randint(min_val, max_val)
            else:
                current_params[param] = random.uniform(min_val, max_val)
        
        # Evaluate the objective function
        current_score = objective_function(current_params)
        
        # Update best if current is better
        if current_score > best_score:  # Change to > if maximizing
            best_score = current_score
            best_params = current_params
    
    return best_params, best_score

In [40]:
gt_val = df_validation.to_dict(orient='records')

In [45]:
import random

def simple_optimize(param_ranges, objective_function, n_iterations=10):
    best_params = None
    best_score = float('-inf')  # Assuming we're minimizing. Use float('-inf') if maximizing.

    for _ in range(n_iterations):
        # Generate random parameters
        current_params = {}
        for param, (min_val, max_val) in param_ranges.items():
            if isinstance(min_val, int) and isinstance(max_val, int):
                current_params[param] = random.randint(min_val, max_val)
            else:
                current_params[param] = random.uniform(min_val, max_val)
        
        # Evaluate the objective function
        current_score = objective_function(current_params)
        
        # Update best if current is better
        if current_score > best_score:  # Change to > if maximizing
            best_score = current_score
            best_params = current_params
    
    return best_params, best_score

In [48]:

def minsearch_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

param_ranges = {
    "task": (0.0, 3.0),
    "instructions": (0.0, 3.0),
    "reasoning": (0.0, 3.0),
    "tags": (0.0, 3.0),
    "category": (0.0, 3.0),
    "difficulty": (0.0, 3.0),
}

def objective(boost_params):
    def search_function(q):
        return minsearch_search(q['question'], boost_params)

    results = evaluate(gt_val, search_function)
    return results['mrr']


In [49]:
simple_optimize(param_ranges, objective, n_iterations=20)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

({'task': 2.2379223274467996,
  'instructions': 2.9498485073533978,
  'reasoning': 1.52224591216505,
  'tags': 1.2705773309268724,
  'category': 1.609700967271582,
  'difficulty': 2.1113138106464224},
 0.716357142857143)

In [50]:
def minsearch_improved(query):
    boost = {
        'exercise_name': 2.11,
        'type_of_activity': 1.46,
        'type_of_equipment': 0.65,
        'body_part': 2.65,
        'type': 1.31,
        'muscle_groups_activated': 2.54,
        'instructions': 0.74
    }

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

evaluate(ground_truth, lambda q: minsearch_improved(q['question']))

  0%|          | 0/1250 [00:00<?, ?it/s]

{'hit_rate': 0.7912, 'mrr': 0.5851914285714294}

## RAG evaluation

In [51]:
prompt2_template = """
You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [52]:
len(ground_truth)

1250

In [53]:
record = ground_truth[0]
question = record['question']

In [54]:
answer_llm = rag(question) 
print(answer_llm)


For your project summary outline, you should include the following key points:

1. **Introduction**: Provide a brief overview of the project, including its purpose and significance.

2. **Achievements**: List the major accomplishments or milestones reached during the project.

3. **Issues**: Identify any challenges or problems encountered throughout the project.

4. **Proposed Solutions**: Suggest solutions to the issues identified, outlining how these can be addressed.

5. **Next Steps**: Propose the next steps or actions that need to be taken moving forward.

6. **Deadlines**: Set timelines for the proposed solutions and next steps to ensure accountability.

By following this structure, you can enhance clarity and make sure essential elements of your project summary are covered.


In [55]:
prompt = prompt2_template.format(question=question, answer_llm=answer_llm)
print(prompt)

You are an expert evaluator for a RAG system.
Your task is to analyze the relevance of the generated answer to the given question.
Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: What key points should I include in my outline for the project summary?
Generated Answer: For your project summary outline, you should include the following key points:

1. **Introduction**: Provide a brief overview of the project, including its purpose and significance.

2. **Achievements**: List the major accomplishments or milestones reached during the project.

3. **Issues**: Identify any challenges or problems encountered throughout the project.

4. **Proposed Solutions**: Suggest solutions to the issues identified, outlining how these can be addressed.

5. **Next Steps**: Propose the next steps or actions that need to be taken moving forward.

6. **Deadlines**: Set timelines for the prop

In [56]:
llm(prompt)

'{\n  "Relevance": "RELEVANT",\n  "Explanation": "The generated answer provides a comprehensive outline for a project summary, addressing essential components such as introduction, achievements, issues, proposed solutions, next steps, and deadlines. These points are directly relevant to what should be included in an outline for a project summary."\n}'

In [57]:
import json
df_sample = df_question.sample(n=200, random_state=1)
sample = df_sample.to_dict(orient='records')

In [60]:
evaluations = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question) 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)

    evaluations.append((record, answer_llm, evaluation))

  0%|          | 0/200 [00:00<?, ?it/s]

In [61]:
df_eval = pd.DataFrame(evaluations, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])


df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [62]:
df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.75
PARTLY_RELEVANT    0.19
NON_RELEVANT       0.06
Name: proportion, dtype: float64

In [63]:
df_eval.to_csv('../data/rag-eval-gpt-4o-mini.csv', index=False)

In [64]:
evaluations_gpt4o = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question, model='gpt-4o') 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)
    
    evaluations_gpt4o.append((record, answer_llm, evaluation))

  0%|          | 0/200 [00:00<?, ?it/s]

In [65]:
df_eval = pd.DataFrame(evaluations_gpt4o, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])

df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [66]:
df_eval.relevance.value_counts()

relevance
RELEVANT           156
PARTLY_RELEVANT     33
NON_RELEVANT        11
Name: count, dtype: int64

In [67]:
df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.780
PARTLY_RELEVANT    0.165
NON_RELEVANT       0.055
Name: proportion, dtype: float64

In [68]:
df_eval.to_csv('../data/rag-eval-gpt-4o.csv', index=False)

In [69]:
model="gpt-5.4-mini"
llm(prompt, model=model)

'{\n  "Relevance": "RELEVANT",\n  "Explanation": "The answer directly addresses how to express gratitude in one sentence and provides a concise example sentence, which matches the user\'s request."\n}'

In [71]:
evaluations_gpt_54mini = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question, model= model) 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt, model='gpt-5.4-mini')
    evaluation = json.loads(evaluation)
    
    evaluations_gpt_54mini.append((record, answer_llm, evaluation))

  0%|          | 0/200 [00:00<?, ?it/s]

In [72]:
df_eval = pd.DataFrame(evaluations_gpt4o, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])

df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [73]:
df_eval.relevance.value_counts()

relevance
RELEVANT           156
PARTLY_RELEVANT     33
NON_RELEVANT        11
Name: count, dtype: int64

In [74]:
df_eval.relevance.value_counts(normalize=True)

relevance
RELEVANT           0.780
PARTLY_RELEVANT    0.165
NON_RELEVANT       0.055
Name: proportion, dtype: float64